# Blood Metrics Regression Analysis

### Dataset Description

The dataset contains RBC (Red Blood Cells), HB (Hemoglobin), HCT (Hematocrit), MCV (Mean Cell Volume), MCH (Mean Cell Hemoglobin), MCHC (Mean Cell Hemoglobin Concentration), RDW (Red Cell Distribution Width), PLT (Platelets), MPV (Mean Platelet Volume), Lymphocytes, Monocytes, Neutrophils, Eosinophils, and Basophils, which are known to be associated with WBC (White Blood Cells). Ignore all other variables.

#### Question 1
- First, load the dataset in Python (for example, using the Pandas library).
- Review the dataset and provide descriptive statistics for each feature and the target variable (WBC).

In [4]:
import pandas as pd

dataset = pd.read_excel("blood-metrics.xlsx")

# Select only the relevant columns
relevant_columns = ['WBC', 'RBC', 'HB', 'HCT', 'MCV', 'MCH', 'MCHC', 'RDW', 'PLT', 'MPV', 'Lenf', 'Monosit', 'Nötrofil', 'Eozinofil', 'Bazofil']
dataset = dataset[relevant_columns]

# Remove rows containing NaN values
dataset.dropna(inplace=True)
question_1 = dataset.describe()
print(question_1)

              WBC         RBC          HB         HCT         MCV         MCH  \
count  999.000000  999.000000  999.000000  999.000000  999.000000  999.000000   
mean     7.902267    4.538835   12.093453   35.665105   79.104234   26.861261   
std      2.181003    0.458546    1.053538    3.127161    7.416157    2.918811   
min      3.500000    3.000000    8.300000   25.200000   51.300000   15.200000   
25%      6.400000    4.300000   11.420000   33.600000   75.000000   25.240000   
50%      7.600000    4.590000   12.000000   35.500000   78.300000   26.500000   
75%      9.000000    4.830000   12.700000   37.400000   81.620000   28.000000   
max     18.340000    6.100000   16.300000   47.100000  103.800000   38.100000   

             MCHC         RDW         PLT         MPV        Lenf     Monosit  \
count  999.000000  999.000000  999.000000  999.000000  999.000000  999.000000   
mean    33.920420   14.406246  240.648949    8.469039    4.311057    0.742784   
std      1.059708    1.5398

#### Question 2
- Prepare the data appropriately to analyze the relationship between the features (RBC, HB, HCT, etc.) and the target variable (WBC). Select the 5 features with the highest correlation with the target variable.

In [9]:
correlation_matrix = dataset.corr()
correlation_wbc = correlation_matrix['WBC'].sort_values(ascending=False)
question_2 = correlation_wbc[1:6].to_string()

top_5_features = correlation_wbc.index[1:6]
print("Top 5 Features with the Highest Correlation:\n", question_2)

Top 5 Features with the Highest Correlation:
 Lenf         0.684340
Nötrofil     0.352815
Monosit      0.265033
Bazofil      0.226184
Eozinofil    0.193125


#### Question 3
- Split the dataset into 80% training and 20% test sets.

In [12]:
from sklearn.model_selection import train_test_split


X = dataset[top_5_features]
y = dataset['WBC']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### Question 4
- Using the Scikit-learn library, build a multiple regression model.
- Create a multiple regression equation showing the effects of the 5 features with the highest correlation to the target variable on WBC.

In [15]:
from sklearn.linear_model import LinearRegression

regressor = LinearRegression()
regressor.fit(X_train, y_train)


equation = "WBC = "
for i, feature in enumerate(top_5_features):
    equation += f"{regressor.coef_[i]:.2f} * {feature} + "
equation += f"{regressor.intercept_:.2f}"
print(equation)

WBC = 0.93 * Lenf + 0.56 * Nötrofil + 0.49 * Monosit + -0.51 * Bazofil + 0.51 * Eozinofil + 1.88


#### Question 5
- Evaluate the model's performance on the training set (R-squared, mean squared error, etc.).
- Evaluate the model on the test set and check its performance.
- Interpret the results and assess the model's success in predicting WBC values.

In [18]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error


y_train_pred = regressor.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
mse_train = mean_squared_error(y_train, y_train_pred)
mae_train = mean_absolute_error(y_train, y_train_pred)


y_test_pred = regressor.predict(X_test)
r2_test = r2_score(y_test, y_test_pred)
mse_test = mean_squared_error(y_test, y_test_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)
results = (
    f"Training Set Performance:\n"
    f"R-square: {r2_train:.2f}\n"
    f"Mean Squared Error: {mse_train:.2f}\n"
    f"Mean Absolute Error: {mae_train:.2f}\n\n"
    f"Test Set Performance:\n"
    f"R-square: {r2_test:.2f}\n"
    f"Mean Squared Error: {mse_test:.2f}\n"
    f"Mean Absolute Error: {mae_test:.2f}\n\n"
    f"Results:\n"
)

threshold_value = 0.70

if abs(r2_train - r2_test) < threshold_value:
    results += (
        "The model's performance on the training and test sets is close to each other. "
        f"Training set R-square: {r2_train:.2f}, Test set R-square: {r2_test:.2f}.\n"
        "This indicates that the model has good generalization ability. "
        "However, further analysis of the test set performance may still be beneficial.\n"
    )
else:
    if r2_train > r2_test:
        if r2_test < threshold_value:
            results += (
                "The model shows good performance on the training set but poor performance on the test set.\n"
                "This situation indicates that the model may be overfitting to the training data and has limited generalization ability.\n"
                f"The high R-square value on the training set ({r2_train:.2f}) suggests that the model fits the training data well, but this does not necessarily translate to good performance on unseen data.\n"
                f"Test set R-square: {r2_test:.2f}\n"
                "Further analysis of the test set performance may still be beneficial.\n"
            )
        else:
            results += (
                "The model shows good performance on the training set but poor performance on the test set.\n"
                "However, further analysis of the test set performance may still be beneficial.\n"
                f"The high R-square value on the training set ({r2_train:.2f}) suggests that the model fits the training data well, but this does not necessarily translate to good performance on unseen data.\n"
                f"Test set R-square: {r2_test:.2f}\n"
            )
print(results)

Training Set Performance:
R-square: 0.63
Mean Squared Error: 1.85
Mean Absolute Error: 0.89

Test Set Performance:
R-square: 0.69
Mean Squared Error: 1.16
Mean Absolute Error: 0.71

Results:
The model's performance on the training and test sets is close to each other. Training set R-square: 0.63, Test set R-square: 0.69.
This indicates that the model has good generalization ability. However, further analysis of the test set performance may still be beneficial.

